In [3]:
import sys
sys.path.append("/scratch/project_465001820/Spatialformer")
import os
os.environ["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"
import scanpy as sc
import spatialformer as sp
import anndata as ad
import numpy as np

/scratch/project_465001820/miniconda3/envs/spatialformer_flash_attn/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/scratch/project_465001820/miniconda3/envs/spatialformer_flash_attn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Download the ".csv" files from the link  
https://drive.google.com/drive/folders/12-vuL-gx_wKiLcZlT_ie_mfZEwiNzeND?usp=drive_link
and store it in your own path

Getting the expression profile

In [1]:
#load cell type exp matrix
ct_train_X = pd.read_csv("/scratch/project_465001820/Spatialformer_main_practice/downstream/cell_types_nich_annotation/data/VUILD110_ct_train_X.csv", index_col = 0)
ct_test_X = pd.read_csv("/scratch/project_465001820/Spatialformer_main_practice/downstream/cell_types_nich_annotation/data/VUILD110_ct_test_X.csv", index_col = 0)
ct_val_X = pd.read_csv("/scratch/project_465001820/Spatialformer_main_practice/downstream/cell_types_nich_annotation/data/VUILD110_ct_val_X.csv", index_col = 0)
 
#lod niche type exp matrix  
nc_train_X = pd.read_csv("/scratch/project_465001820/Spatialformer_main_practice/downstream/cell_types_nich_annotation/data/VUILD110_nc_train_X.csv", index_col = 0)
nc_test_X = pd.read_csv("/scratch/project_465001820/Spatialformer_main_practice/downstream/cell_types_nich_annotation/data/VUILD110_nc_test_X.csv", index_col = 0)
nc_val_X = pd.read_csv("/scratch/project_465001820/Spatialformer_main_practice/downstream/cell_types_nich_annotation/data/VUILD110_nc_val_X.csv", index_col = 0)


NameError: name 'pd' is not defined

Getting the cell type annotations

In [ ]:
def get_dataset(X):
    """
    Getting the dataset for the leave -ut sample - "VUILD110"
    This dataset will be used to calculate the embeddings and the downstream cell types/niches annotations

    Parameters:
    X: The expression matrix with cell x gene matrix that comes from the leave-out sample/slide.

    """
    sample_name = "VUILD110"
    cell_ids = list(X.index)
    #loading the dataset
    datapath = "/scratch/project_465001820/Spatialformer_main_practice/cache/" # Customise your own path here to store the cache of the dataset, otherwise, it will be saved to the root dir.
    combined_dataset = load_dataset("TerminatorJ/xenium_pandavid_dataset4", cache_dir = datapath, num_proc=8)
    index_path = "/scratch/project_465001820/Spatialformer_main_practice/data/sample_cell_index.pkl"
    sample_cell_index = get_index(combined_dataset, save_file = index_path)
    index = [sample_cell_index[sample_name][cell_id] for cell_id in cell_ids]

    combined_dataset_all = concatenate_datasets([combined_dataset["train"], combined_dataset["test"], combined_dataset["validation"]])
    dataset = combined_dataset_all.select(index)
    return dataset

#get train test val dataset
train_ct_dataset = get_dataset(ct_train_X)
test_ct_dataset = get_dataset(ct_test_X)
val_ct_dataset = get_dataset(ct_val_X)

train_nc_dataset = get_dataset(nc_train_X)
test_nc_dataset = get_dataset(nc_test_X)
val_nc_dataset = get_dataset(nc_val_X)



Building the AnnData for getting the embeddings

In [ ]:
#For expression matrices
ct_train_adata = ad.AnnData(
    X=ct_train_X,
    obs=pd.DataFrame(index=ct_train_X.index),
    var=pd.DataFrame(index=ct_train_X.columns),
)

ct_test_adata = ad.AnnData(
    X=ct_test_X,
    obs=pd.DataFrame(index=ct_test_X.index),
    var=pd.DataFrame(index=ct_test_X.columns),
)

ct_val_adata = ad.AnnData(
    X=ct_val_X,
    obs=pd.DataFrame(index=ct_val_X.index),
    var=pd.DataFrame(index=ct_val_X.columns),
)

nc_train_adata = ad.AnnData(
    X=nc_train_X,
    obs=pd.DataFrame(index=nc_train_X.index),
    var=pd.DataFrame(index=nc_train_X.columns),
)

nc_test_adata = ad.AnnData(
    X=nc_test_X,
    obs=pd.DataFrame(index=nc_test_X.index),
    var=pd.DataFrame(index=nc_test_X.columns),
)

nc_val_adata = ad.AnnData(
    X=nc_val_X,
    obs=pd.DataFrame(index=nc_val_X.index),
    var=pd.DataFrame(index=nc_val_X.columns),
)

#For annotations
#For cell types
ct_train_adata.obs["Annotations"] = train_ct_dataset["Annotations"]
ct_test_adata.obs["Annotations"] = test_ct_dataset["Annotations"]
ct_val_adata.obs["Annotations"] = val_ct_dataset["Annotations"]

nc_train_adata.obs["Niche_Annotations"] = train_nc_dataset["Niche_Annotations"]
nc_test_adata.obs["Niche_Annotations"] = test_nc_dataset["Niche_Annotations"]
nc_val_adata.obs["Niche_Annotations"] = val_nc_dataset["Niche_Annotations"]

#save to local
ct_train_adata.write_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_train_adata.h5ad")
ct_test_adata.write_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_test_adata.h5ad")
ct_val_adata.write_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_val_adata.h5ad")

nc_train_adata.write_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_train_adata.h5ad")
nc_test_adata.write_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_test_adata.h5ad")
nc_val_adata.write_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_val_adata.h5ad")


/tmp/ipykernel_48043/4273971423.py:2: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ct_train_adata = ad.AnnData(
/tmp/ipykernel_48043/4273971423.py:8: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ct_test_adata = ad.AnnData(
/tmp/ipykernel_48043/4273971423.py:14: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ct_val_adata = ad.AnnData(
/tmp/ipykernel_48043/4273971423.py:20: FutureWar

Loading the AnnData

Download the ".h5ad" files from the link  
https://drive.google.com/drive/folders/12-vuL-gx_wKiLcZlT_ie_mfZEwiNzeND?usp=drive_link
and store it in your own path

In [4]:
ct_train_adata = ad.read_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_train_adata.h5ad")
ct_train_adata.var["gene_name"] = ct_train_adata.var.index
ct_test_adata = ad.read_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_test_adata.h5ad")
ct_test_adata.var["gene_name"] = ct_test_adata.var.index
ct_val_adata = ad.read_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_val_adata.h5ad")
ct_val_adata.var["gene_name"] = ct_val_adata.var.index

nc_train_adata = ad.read_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_train_adata.h5ad")
nc_train_adata.var["gene_name"] = nc_train_adata.var.index
nc_test_adata = ad.read_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_test_adata.h5ad")
nc_test_adata.var["gene_name"] = nc_test_adata.var.index
nc_val_adata = ad.read_h5ad("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_val_adata.h5ad")
nc_val_adata.var["gene_name"] = nc_val_adata.var.index

# Getting the cell type embeddings for SpatialFormer

Loading the model

- The checkpoint of the singular input model can be downloaded via: https://figshare.com/articles/dataset/single_input/28452209?file=52503695)

In [16]:
%%time
batch_size = 32
#configuring the model
model_ckp_path = "/scratch/project_465001820/Spatialformer/output/checkpoints/stepstep=0176000-traintrain_total_loss=-2.7789-valval_total_loss=0.0000.ckpt"
# model_ckp_path = "/scratch/project_465001820/Spatialformer/output/checkpoints/stepstep=0194000-traintrain_total_loss=-3.2361-valval_total_loss=0.0000.ckpt"
# model_ckp_path = "/scratch/project_465001820/Spatialformer/output/checkpoints/stepstep=0192000-traintrain_total_loss=-4.1449-valval_total_loss=0.0000.ckpt"

tissue = "Lung"
condition = "Disease"
method = "gene" #getting the cls token embeddings
max_len = None
ct_train_embed_adata = sp.tl.embed_data(adata = ct_train_adata, 
                              tissue = tissue,
                              condition = condition,
                              method = method,
                              model_ckp_path = model_ckp_path, 
                              batch_size = batch_size,
                              mode = "single",
                              threshold = 0.7,
                              num_workers = 16,
                              gene_median_path = "/scratch/project_465001820/Spatialformer/data/gene_median.pkl",
                              max_len=max_len
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


Spatialformer - INFO - Setting the model to evaluation mode...
Spatialformer - INFO - Model mapped to device: cuda
Spatialformer - INFO - Encoding data into batches...


Generating embeddings:   0%|          | 0/2504 [00:00<?, ?it/s]

running max length: 112


...

running max length: 150


Generating embeddings:  51%|█████▏    | 1285/2504 [01:39<01:19, 15.43it/s]

running max length: 167


Generating embeddings: 100%|██████████| 2504/2504 [03:01<00:00, 13.83it/s]


CPU times: user 16min 25s, sys: 6.76 s, total: 16min 32s
Wall time: 3min 6s


In [17]:

ct_test_embed_adata = sp.tl.embed_data(adata = ct_test_adata, 
                              tissue = tissue,
                              condition = condition,
                              method = method,
                              model_ckp_path = model_ckp_path, 
                              batch_size = batch_size,
                              mode = "single",
                              threshold = 0.7,
                              num_workers = 8,
                              gene_median_path = "/scratch/project_465001820/Spatialformer/data/gene_median.pkl",
                              max_len=max_len
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


Spatialformer - INFO - Setting the model to evaluation mode...
Spatialformer - INFO - Model mapped to device: cuda
Spatialformer - INFO - Encoding data into batches...


Generating embeddings:   1%|          | 3/313 [00:03<05:04,  1.02it/s]

running max length: 114
running max length: 118
running max length: 121
running max length: 125


...

Generating embeddings:  16%|█▌        | 49/313 [00:06<00:17, 15.09it/s]

running max length: 133
running max length: 145


Generating embeddings:  69%|██████▉   | 217/313 [00:17<00:07, 12.02it/s]

running max length: 153


Generating embeddings: 100%|██████████| 313/313 [00:23<00:00, 13.06it/s]


In [18]:
ct_val_embed_adata = sp.tl.embed_data(adata = ct_val_adata, 
                              tissue = tissue,
                              condition = condition,
                              method = method,
                              model_ckp_path = model_ckp_path, 
                              batch_size = batch_size,
                              mode = "single",
                              threshold = 0.7,
                              num_workers = 8,
                              gene_median_path = "/scratch/project_465001820/Spatialformer/data/gene_median.pkl"
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


Spatialformer - INFO - Setting the model to evaluation mode...
Spatialformer - INFO - Model mapped to device: cuda
Spatialformer - INFO - Encoding data into batches...


Generating embeddings:   1%|          | 3/314 [00:05<06:48,  1.31s/it]

running max length: 120


...

Generating embeddings:  12%|█▏        | 39/314 [00:07<00:19, 14.27it/s]

running max length: 134


Generating embeddings:  26%|██▌       | 81/314 [00:10<00:15, 15.07it/s]

running max length: 147


Generating embeddings: 100%|██████████| 314/314 [00:27<00:00, 11.53it/s]


In [19]:
nc_train_embed_adata = sp.tl.embed_data(adata = nc_train_adata, 
                              tissue = tissue,
                              condition = condition,
                              method = method,
                              model_ckp_path = model_ckp_path, 
                              batch_size = batch_size,
                              mode = "single",
                              threshold = 0.7,
                              num_workers = 8,
                              gene_median_path = "/scratch/project_465001820/Spatialformer/data/gene_median.pkl",
                              max_len=max_len
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


Spatialformer - INFO - Setting the model to evaluation mode...
Spatialformer - INFO - Model mapped to device: cuda
Spatialformer - INFO - Encoding data into batches...


Generating embeddings:   0%|          | 1/2504 [00:04<3:14:05,  4.65s/it]

running max length: 101
running max length: 105


...

running max length: 137


Generating embeddings:  13%|█▎        | 319/2504 [00:26<02:38, 13.78it/s]

running max length: 143


Generating embeddings:  14%|█▎        | 340/2504 [00:28<02:59, 12.02it/s]


KeyboardInterrupt: 

In [ ]:
nc_test_embed_adata = sp.tl.embed_data(adata = nc_test_adata, 
                              tissue = tissue,
                              condition = condition,
                              method = method,
                              model_ckp_path = model_ckp_path, 
                              batch_size = batch_size,
                              mode = "single",
                              threshold = 0.7,
                              num_workers = 8,
                              gene_median_path = "/scratch/project_465001820/Spatialformer/data/gene_median.pkl",
                              max_len=max_len
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


Spatialformer - INFO - Setting the model to evaluation mode...
Spatialformer - INFO - Model mapped to device: cuda
Spatialformer - INFO - Encoding data into batches...


Generating embeddings:   1%|          | 3/313 [00:03<04:39,  1.11it/s]

running max length: 56


Generating embeddings: 100%|██████████| 313/313 [00:19<00:00, 16.42it/s]


In [ ]:
nc_val_embed_adata = sp.tl.embed_data(adata = nc_val_adata, 
                              tissue = tissue,
                              condition = condition,
                              method = method,
                              model_ckp_path = model_ckp_path, 
                              batch_size = batch_size,
                              mode = "single",
                              threshold = 0.7,
                              num_workers = 8,
                              gene_median_path = "/scratch/project_465001820/Spatialformer/data/gene_median.pkl",
                              max_len=max_len
                            )

Spatialformer - INFO - Loading the SpatialFormer model...


[rank: 0] Global seed set to 42


Spatialformer - INFO - Setting the model to evaluation mode...
Spatialformer - INFO - Model mapped to device: cuda
Spatialformer - INFO - Encoding data into batches...


Generating embeddings:   1%|          | 3/313 [00:03<05:04,  1.02it/s]

running max length: 56


Generating embeddings: 100%|██████████| 313/313 [00:20<00:00, 14.92it/s]


Saving the cell type embeddings results

In [11]:
ct_train_embed_adata.obsm["X_SpaF"].shape

(80119, 512)

In [20]:
np.save(f"/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_single_ct_concat_train_embed_5k_{method}_176000_{max_len}.npy",ct_train_embed_adata.obsm["X_SpaF"])
np.save("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_train_ct.npy", ct_train_embed_adata.obs["Annotations"].to_numpy())

np.save(f"/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_single_ct_concat_test_embed_5k_{method}_176000_{max_len}.npy",ct_test_embed_adata.obsm["X_SpaF"])
np.save("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_test_ct.npy", ct_test_embed_adata.obs["Annotations"].to_numpy())

np.save(f"/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_single_ct_concat_val_embed_5k_{method}_176000_{max_len}.npy",ct_val_embed_adata.obsm["X_SpaF"])
np.save("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_val_ct.npy", ct_val_embed_adata.obs["Annotations"].to_numpy())

Saving the cell niches embeddings

In [15]:
np.save(f"/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_single_nc_concat_train_embed_5k_{method}_194000_{max_len}.npy",nc_train_embed_adata.obsm["X_SpaF"])
np.save("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_train_nc.npy", nc_train_embed_adata.obs["Niche_Annotations"].to_numpy())

np.save(f"/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_single_nc_concat_test_embed_5k_{method}_194000_{max_len}.npy",nc_test_embed_adata.obsm["X_SpaF"])
np.save("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_test_nc.npy", nc_test_embed_adata.obs["Niche_Annotations"].to_numpy())

np.save(f"/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_single_nc_concat_val_embed_5k_{method}_194000_{max_len}.npy",nc_val_embed_adata.obsm["X_SpaF"])
np.save("/scratch/project_465001820/Spatialformer/downstream/cell_types_nich_annotation/data/spa_val_nc.npy", nc_val_embed_adata.obs["Niche_Annotations"].to_numpy())